#   Clean_text

In [ ]:
import re
import unicodedata
import html

# Định nghĩa các Regex bên ngoài hàm để tối ưu hiệu năng
RE_SCRIPT_STYLE = re.compile(r'<(script|style)[^>]*>.*?</\1>', flags=re.IGNORECASE | re.DOTALL)
RE_HTML_TAGS = re.compile(r'<[^>]+>')
RE_URLS = re.compile(r'http[s]?://\S+|www\.\S+')
RE_EMAILS = re.compile(r'\S+@\S+\.\S+')
RE_INVISIBLE = re.compile(r'[\u200B-\u200D\uFEFF\u200E\u200F]')
RE_SPACES = re.compile(r'\s+')

def clean_text(text: str, remove_urls: bool = True, remove_emails: bool = True) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    # 1. Giải mã các ký tự HTML (ví dụ &amp; -> &)
    text = html.unescape(text)
    
    # 2. Loại bỏ Script, Style và Tags HTML
    text = RE_SCRIPT_STYLE.sub(' ', text)
    text = RE_HTML_TAGS.sub(' ', text)

    # 3. Loại bỏ URLs và Emails nếu được yêu cầu
    if remove_urls:
        text = RE_URLS.sub('', text)
    if remove_emails:
        text = RE_EMAILS.sub('', text)

    # 4. Chuẩn hóa Unicode (NFKC giúp gom các ký tự tổ hợp lại)
    text = unicodedata.normalize('NFKC', text)
    
    # 5. Loại bỏ ký tự tàng hình
    text = RE_INVISIBLE.sub('', text)

    # 6. LỌC KÝ TỰ ĐẶC BIỆT (⭕️, Emoji, Biểu tượng lạ)
    # L: Letter (Chữ cái tiếng Việt, Lào, Thái, Anh...)
    # N: Number (Số)
    # P: Punctuation (Dấu câu như , . : ; ?)
    # Z: Separator (Khoảng trắng)
    # M: Mark (Các dấu thanh, dấu phụ - CỰC KỲ QUAN TRỌNG cho tiếng Lào/Thái)
    allowed_categories = {'L', 'N', 'P', 'Z', 'M'}
    
    clean_chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        # Giữ lại nếu thuộc nhóm cho phép VÀ không phải ký tự điều khiển (C)
        if cat[0] in allowed_categories and cat[0] != 'C':
            clean_chars.append(ch)
        # Giữ lại xuống dòng hoặc tab nếu bạn muốn (tùy chọn)
        elif ch in ['\n', '\t']:
            clean_chars.append(' ')

    text = "".join(clean_chars)

    # 7. Xử lý khoảng trắng thừa
    text = RE_SPACES.sub(' ', text)

    return text.strip()

# --- Kiểm tra thử ---
sample = "⭕️ ກຸຫຼາບສີແດງ: ຫມາຍເຖິງຄວາມຮັກ, ຄວາມປາດຖະຫນາ 📧 test@gmail.com"
print(clean_text(sample)) 
# Kết quả: "ກຸຫຼາບສີແດງ: ຫມາຍເຖິງຄວາມຮັກ, ຄວາມປາດຖະຫນາ"

# Craw_data

In [ ]:
from datasets import load_dataset
import json
from tqdm import tqdm

languages = {
    # "lo":  "lao_Laoo",
    # "id": "ind_Latn",
    # "ms": "zsm_Latn",
    # "fil": "fil_Latn",
    "khm": "khm_Khmr",
    "th":  "tha_Thai",
    # "my":  "mya_Mymr",
    # "vi":  "vie_Latn"
}

output_file = "sea_translation.jsonl"
limit_per_lang = 60000
max_words = 1000000000000000 
min_words = 5 

def is_strict_valid(x, lang):

    # if lang == "lo":
    #     return True
    src_text = x["og_full_text"]
    tgt_text = x["translated_text"]
    
    #1. Kiểm tra độ dài cơ bản (số từ)
    src_words = len(src_text.split())
    tgt_words = len(tgt_text.split())
    if not (min_words <= src_words <= max_words and min_words <= tgt_words <= max_words):
        return False

    # 2. Lọc theo điểm ngôn ngữ (og_language_score)
    # Thường nên để > 0.98 để đảm bảo văn bản sạch, không lẫn ngôn ngữ khác
    if x["og_language_score"] < 0.98:
        return False

    # 3. Lọc theo điểm chất lượng giáo dục (edu_score)
    # Thang điểm 0-3. 
    # 2: Nội dung tốt, có thông tin. 3: Nội dung giáo dục cao, chuyên nghiệp.
    if x["edu_score"] < 2 and lang != "lo":
        return False

    # 4. Kiểm tra sự tương xứng độ dài (Token Count Ratio)
    # Tránh trường hợp dịch thiếu hoặc dịch lặp (hallucination)
    og_tokens = x["og_token_count"]
    tr_tokens = x["translated_token_count"]
    if og_tokens > 0 and lang != "lo":
        ratio = tr_tokens / og_tokens
        # Nếu bản dịch dài gấp đôi hoặc ngắn hơn một nửa bản gốc thì loại
        if ratio < 0.5 or ratio > 2.0:
            return False
        
    if x["minhash_cluster_size"] > 10:
        return False
            
    return True

with open(output_file, "w", encoding="utf8") as f:
    for lang, config in languages.items():
        print(f"\n--- Loading {lang} ({config}) ---")

        try:
            ds = load_dataset(
                "HuggingFaceFW/finetranslations",
                name=config,
                split="train",
                streaming=True
            )
        except Exception as e:
            print(f"Error loading {lang}: {e}")
            continue

        count = 0
        pbar = tqdm(total=limit_per_lang, desc=f"Processing {lang}")

        for x in ds:
            if is_strict_valid(x, lang):
                data = {
                    "source_lang": lang,
                    "source": clean_text(x["og_full_text"].strip()),
                    "target": clean_text(x["translated_text"].strip()),
                }

                f.write(json.dumps(data, ensure_ascii=False) + "\n")
                count += 1
                pbar.update(1)

            if count >= limit_per_lang:
                break

        pbar.close()
        print(f"Finished {lang}: Collected {count} high-quality samples")

# Thống Kê

In [ ]:
import json
from collections import defaultdict

stats = defaultdict(lambda: {
    "sentences": 0,
    "words": 0,
    "max_len": 0,
    "min_len": 10**9,
    "max_len_target" : 0
})

file_path = r"clustered_results1/news.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:

        data = json.loads(line.strip())

        lang = data["source_lang"]
        src = data["source"]
        tgt = data["target"]

        if not src or not tgt:
            continue

        # đếm số từ
        src_len = len(src.split())
        tgt_len = len(tgt.split())

        # source language
        stats[lang]["sentences"] += 1
        stats[lang]["words"] += src_len
        stats[lang]["max_len"] = max(stats[lang]["max_len"], src_len)
        stats[lang]["max_len_target"] = max(stats[lang]["max_len_target"], tgt_len)
        stats[lang]["min_len"] = min(stats[lang]["min_len"], src_len)

        # english target
        # stats["en"]["sentences"] += 1
        # stats["en"]["words"] += tgt_len
        # stats["en"]["max_len"] = max(stats["en"]["max_len"], tgt_len)
        # stats["en"]["min_len"] = min(stats["en"]["min_len"], tgt_len)


print(f"{'Lang':<10} | {'#Sentences':>10} | {'#Words':>10} | {'Avg words':>12} | {'Max':>6} | {'Min':>6}")
print("-" * 70)

for lang, s in stats.items():

    avg_len = s["words"] / s["sentences"]

    print(
        f"{lang:<10} | "
        f"{s['sentences']:>10} | "
        f"{s['words']:>10} | "
        f"{avg_len:>12.2f} | "
        f"{s['max_len']:>6} | "
        f"{s['min_len']:>6}"
        F"{s['max_len_target']:>6}"
    )

In [ ]:
import os
import json
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# CẤU HÌNH THÔNG TIN
# ==========================================
# Thư mục chứa các file cluster (phải khớp với OUTPUT_DIR ở code trước)
INPUT_DIR = 'clustered_results1' 
# Tên file biểu đồ xuất ra
OUTPUT_CHART = 'domain_distribution.png'

def plot_domain_distribution():
    print(f"1. Đang đọc thư mục: {INPUT_DIR}...")
    
    if not os.path.exists(INPUT_DIR):
        print(f"Lỗi: Thư mục {INPUT_DIR} không tồn tại!")
        return

    domain_counts = {}
    
    # 2. Lấy danh sách các file .jsonl
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.jsonl')]
    
    if not files:
        print("Không tìm thấy file .jsonl nào trong thư mục.")
        return

    print(f"2. Đang thống kê số lượng bản ghi trong {len(files)} file...")
    for filename in tqdm(files, desc="Counting"):
        filepath = os.path.join(INPUT_DIR, filename)
        
        # Đếm số dòng trong mỗi file
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                count = sum(1 for line in f if line.strip())
            
            # Sử dụng tên file (bỏ đuôi .jsonl) làm tên domain
            domain_name = filename.replace('.jsonl', '')
            domain_counts[domain_name] = count
        except Exception as e:
            print(f"Lỗi khi đọc file {filename}: {e}")

    # 3. Chuẩn bị dữ liệu để vẽ
    # Sắp xếp để biểu đồ trông đẹp hơn (từ lớn đến nhỏ)
    sorted_domains = dict(sorted(domain_counts.items(), key=lambda item: item[1], reverse=True))
    
    labels = list(sorted_domains.keys())
    sizes = list(sorted_domains.values())
    
    # 4. Vẽ biểu đồ tròn
    print("3. Đang tạo biểu đồ...")
    plt.figure(figsize=(12, 10))
    
    # Tùy chỉnh màu sắc
    colors = plt.get_cmap('tab20')(range(len(labels)))
    
    # Vẽ pie chart
    patches, texts, autotexts = plt.pie(
        sizes, 
        labels=labels, 
        autopct='%1.1f%%', 
        startangle=140,
        colors=colors,
        pctdistance=0.85, # Vị trí của số %
        explode=[0.05 if i < 3 else 0 for i in range(len(labels))] # Đẩy các cụm lớn nhất ra một chút
    )

    # Làm đẹp text
    for text in texts:
        text.set_fontsize(9)
    for autotext in autotexts:
        autotext.set_fontsize(8)
        autotext.set_color('white')
        autotext.set_weight('bold')

    plt.title('Thống kê tỷ lệ các Domain (Clusters)', fontsize=15, pad=20)
    plt.axis('equal')  # Đảm bảo hình tròn không bị méo

    # Thêm chú thích bên cạnh nếu có quá nhiều cụm
    if len(labels) > 10:
        plt.legend(patches, labels, loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

    # 5. Lưu và hiển thị
    plt.tight_layout()
    plt.savefig(OUTPUT_CHART, dpi=300, bbox_inches='tight')
    print(f"✅ Đã lưu biểu đồ tại: {OUTPUT_CHART}")
    plt.show()

    # In ra bảng thống kê nhanh
    print("\n--- BẢNG THỐNG KÊ CHI TIẾT ---")
    print(f"{'Domain':<25} | {'Số lượng':<10} | {'Tỷ lệ':<10}")
    total = sum(sizes)
    for label, size in sorted_domains.items():
        percent = (size / total) * 100
        print(f"{label:<25} | {size:<10} | {percent:>6.2f}%")

if __name__ == "__main__":
    plot_domain_distribution()

# Tách câu

In [ ]:
import json
import icu
import concurrent.futures
from itertools import islice
import time
import re

LANG_MAP = {"id": "id_ID", "ms": "ms_MY", "th": "th_TH", "fil": "fil_PH", "khm": "km_KH", "lo": "lo_LA", "my": "my_MM", "vi": "vi_VN", "en": "en_US"}
CHUNK_SIZE = 100000
CPU_WORKERS = 18

def clean_text(text):
    if not text: return ""
    text = re.sub(r'\s+', ' ', text)
    return text.replace('\u200b', '').replace('\ufeff', '').strip()

def segment_sentences(text, lang_code):
    text = clean_text(text)
    if not text: return []
    locale = icu.Locale(LANG_MAP.get(lang_code, "en_US"))
    boundary = icu.BreakIterator.createSentenceInstance(locale)
    boundary.setText(text)
    sentences = []
    start = boundary.first()
    for end in boundary:
        sentence = text[start:end].strip()
        if sentence and len(sentence) > 1: sentences.append(sentence)
        start = end
    return sentences

# ==========================================
# THUẬT TOÁN ALIGNMENT QUY HOẠCH ĐỘNG (DP)
# ==========================================
def align_by_length_dp(src_sents, tgt_sents):
    if not src_sents or not tgt_sents: return []

    total_src_len = sum(len(s) for s in src_sents)
    total_tgt_len = sum(len(s) for s in tgt_sents)
    if total_src_len == 0 or total_tgt_len == 0: return []
    
    # Tỷ lệ độ dài kỳ vọng
    R = total_tgt_len / total_src_len 

    # Các kịch bản ghép câu: (src, tgt) -> (1,1), (2,1), (1,2), (2,2), bỏ qua (1,0), (0,1)
    steps = [(1, 1), (2, 1), (1, 2), (1, 0), (0, 1), (2, 2)]
    dp = {(0, 0): 0}
    backtrack = {}

    def get_cost(src_idx, tgt_idx, src_count, tgt_count):
        if src_count == 0: return len(tgt_sents[tgt_idx - 1]) * 2 # Phạt nặng nếu bỏ câu Target
        if tgt_count == 0: return len(src_sents[src_idx - 1]) * R * 2 # Phạt nặng nếu bỏ câu Source

        src_len = sum(len(src_sents[i]) for i in range(src_idx - src_count, src_idx))
        tgt_len = sum(len(tgt_sents[i]) for i in range(tgt_idx - tgt_count, tgt_idx))
        
        expected_tgt_len = src_len * R
        return abs(tgt_len - expected_tgt_len)

    for i in range(len(src_sents) + 1):
        for j in range(len(tgt_sents) + 1):
            if i == 0 and j == 0: continue

            best_cost = float('inf')
            best_step = None

            for di, dj in steps:
                if i - di >= 0 and j - dj >= 0 and (i - di, j - dj) in dp:
                    cost = dp[(i - di, j - dj)] + get_cost(i, j, di, dj)
                    # Ưu tiên ghép 1-1 bằng cách thêm điểm phạt nhẹ cho các bước gộp câu
                    penalty = 0 if (di == 1 and dj == 1) else 15
                    total_cost = cost + penalty

                    if total_cost < best_cost:
                        best_cost = total_cost
                        best_step = (di, dj)

            if best_step:
                dp[(i, j)] = best_cost
                backtrack[(i, j)] = best_step

    # Truy vết ngược để lấy kết quả cặp câu
    aligned_pairs = []
    i, j = len(src_sents), len(tgt_sents)
    while i > 0 or j > 0:
        if (i, j) not in backtrack: break
        di, dj = backtrack[(i, j)]

        if di > 0 and dj > 0:
            src_matched = " ".join(src_sents[i - di: i])
            tgt_matched = " ".join(tgt_sents[j - dj: j])
            aligned_pairs.append((src_matched, tgt_matched))

        i -= di
        j -= dj

    aligned_pairs.reverse()
    return aligned_pairs

def parse_and_segment(line):
    try:
        data = json.loads(line)
        src_lang = data.get("source_lang", "id")
        src_sents = segment_sentences(data.get("source", ""), src_lang)
        tgt_sents = segment_sentences(data.get("target", ""), "en")
        
        # Dùng thuật toán DP để gióng hàng thay vì zip mù quáng
        aligned_pairs = align_by_length_dp(src_sents, tgt_sents)
        
        return {
            "source_lang": src_lang,
            "pairs": aligned_pairs
        }
    except Exception:
        return None

def process_segmentation(input_file, output_file):
    print(f"Bắt đầu tách và gióng hàng câu từ file: {input_file}...")
    start_time = time.time()
    total_processed = 0
    total_sentences = 0

    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        while True:
            lines_chunk = list(islice(f_in, CHUNK_SIZE))
            if not lines_chunk: break
                
            with concurrent.futures.ProcessPoolExecutor(max_workers=CPU_WORKERS) as executor:
                results = executor.map(parse_and_segment, lines_chunk)

                for res in results:
                    if not res: continue
                    src_lang = res["source_lang"]

                    for src, tgt in res["pairs"]:
                        row = {"source_lang": src_lang, "source": src, "target": tgt}
                        f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
                        total_sentences += 1
            
            total_processed += len(lines_chunk)
            print(f"Đã xử lý {total_processed} văn bản -> {total_sentences} cặp câu đã gióng hàng")

    print(f"Hoàn thành trong {time.time() - start_time:.2f} giây!")

if __name__ == '__main__':
    IN_FILE = "sea_translation.jsonl"
    OUTPUT_FILE = "segmented_data_clean.jsonl"
    process_segmentation(IN_FILE, OUTPUT_FILE)

# Ngôn ngữ

In [ ]:
import json
import fasttext

# Load model (Giả định bạn đã tải file lid.176.ftz từ bước trước)
model_path = "lid.176.ftz"
print("Đang tải model vào bộ nhớ...")
model = fasttext.load_model(model_path)

lang_map = {
    "id": "id",
    "ms": "ms",
    "tl": "fil",  
    "km": "khm",  
    "lo": "lo",
    "th": "th",
    "my": "my",
    "en": "en"
}
target_langs = set(lang_map.keys())

input_file = "segmented_data_khm_th.jsonl"
output_file = "segmented_data_khm_th_clean.jsonl"

print("Đang xử lý dữ liệu với bộ lọc độ tin cậy và độ dài...")
count_removed_short = 0
count_removed_lang = 0
count_total = 0

with open(input_file, 'r', encoding='utf-8') as infile, \
     open(output_file, 'w', encoding='utf-8') as outfile:
    
    for line in infile:
        line = line.strip()
        if not line:
            continue
            
        try:
            data = json.loads(line)
            source_text = data.get("source", "")
            original_lang = data.get("source_lang", "")
            target_text = data.get("target", "")
            
            if not source_text:
                continue

            text_clean = source_text.replace('\n', ' ')
            text_len = target_text.replace('\n', ' ')
            word_count = len(text_len.split())

            # --- TÍNH NĂNG 1: Loại các câu có độ dài < 3 từ ---
            if word_count < 4 or word_count > 100:
                count_removed_short += 1
                continue

            # Dự đoán ngôn ngữ
            predictions = model.predict(text_clean, k=1)
            raw_detected_lang = predictions[0][0].replace("__label__", "")
            confidence = predictions[1][0]
            
            # --- TÍNH NĂNG 2: Kiểm tra ngôn ngữ có thuộc lang_map không ---
            if raw_detected_lang not in target_langs:
                count_removed_lang += 1
                continue

            detected_lang = lang_map.get(raw_detected_lang, raw_detected_lang)
            final_lang = detected_lang
            
            # --- BẮT ĐẦU CÁC LUẬT BẢO VỆ NHÃN GỐC ---
                
            # 2. Xử lý kẹt id/ms: Yêu cầu độ tự tin > 85% mới cho phép đổi qua lại
            # if original_lang in ['id', 'ms'] and detected_lang in ['id', 'ms']:
            #     if confidence < 0.85:
            #         final_lang = original_lang
            
            # 3. Các trường hợp còn lại: Yêu cầu độ tự tin > 50% để tránh AI đoán nhiễu
            if confidence < 0.7 and original_lang in target_langs:
                final_lang = original_lang
                
            # --- KẾT THÚC CÁC LUẬT ---

            # Cập nhật nhãn và ghi vào file
            data["source_lang"] = final_lang
            json.dump(data, outfile, ensure_ascii=False)
            outfile.write('\n')
            count_total += 1
            
        except Exception as e:
            print(f"Lỗi: {e} - Dòng: {line}")

print("-" * 30)
print(f"✅ Đã hoàn thành!")
print(f"   - Số câu bị loại do quá ngắn (<3 từ): {count_removed_short}")
print(f"   - Số câu bị loại do không thuộc target langs: {count_removed_lang}")
print(f"   - Số câu hợp lệ đã lưu: {count_total}")
print(f"📂 Kết quả tại: {output_file}")

# Ngữ nghĩa với LaBSE

In [ ]:
import json
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

a = 0.7
LANGUAGE_THRESHOLDS = {
    "id": a,
    "ms": a,
    "fil": a,
    "khm": a,
    "lo": a,
    "th": a,
    "my": a,
    "vi": a
}


def apply_labse_filter_optimized(
    input_file,
    output_file,
    batch_size=2048,
    model_batch_size=256,
    default_threshold=0.8
):
    """
    Lọc dữ liệu song ngữ bằng LaBSE tối ưu cho GPU.
    Phù hợp với dataset vài triệu câu.
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Đang tải LaBSE trên {device} ...")

    model = SentenceTransformer(
        "sentence-transformers/LaBSE",
        device=device
    )

    kept = 0
    removed = 0

    data_batch = []
    src_batch = []
    tgt_batch = []
    threshold_batch = []

    def process_batch(fout):
        nonlocal kept, removed

        if not data_batch:
            return

        with torch.no_grad():
            all_texts = src_batch + tgt_batch

            embeddings = model.encode(
                all_texts,
                batch_size=model_batch_size,
                convert_to_tensor=True,
                normalize_embeddings=True,
                show_progress_bar=False
            )

            n = len(src_batch)
            emb_src = embeddings[:n]
            emb_tgt = embeddings[n:]

            # cosine similarity nhanh hơn vì đã normalize
            scores = (emb_src * emb_tgt).sum(dim=1)

            thresholds = torch.tensor(
                threshold_batch,
                device=scores.device
            )

            keep_mask = scores > thresholds

        output_lines = []

        for item, keep_flag in zip(data_batch, keep_mask.tolist()):
            if keep_flag:
                output_lines.append(json.dumps(item, ensure_ascii=False))
                kept += 1
            else:
                removed += 1

        if output_lines:
            fout.write("\n".join(output_lines) + "\n")

    print(f"Bắt đầu xử lý: {input_file}")

    with open(input_file, "r", encoding="utf-8") as fin, \
         open(output_file, "w", encoding="utf-8") as fout:

        for line in tqdm(fin, desc="Filtering"):
            try:
                data = json.loads(line)

                src = data.get("source", "").strip()
                tgt = data.get("target", "").strip()
                lang = data.get("source_lang", "")

                if not src or not tgt:
                    removed += 1
                    continue

                threshold = LANGUAGE_THRESHOLDS.get(lang, default_threshold)

                data_batch.append(data)
                src_batch.append(src)
                tgt_batch.append(tgt)
                threshold_batch.append(threshold)

                if len(data_batch) >= batch_size:
                    process_batch(fout)

                    data_batch.clear()
                    src_batch.clear()
                    tgt_batch.clear()
                    threshold_batch.clear()

            except Exception:
                removed += 1

        # xử lý phần còn lại
        process_batch(fout)

    print("\n--- HOÀN THÀNH ---")
    print(f"Giữ lại : {kept:,}")
    print(f"Loại bỏ : {removed:,}")
    print(f"Output  : {output_file}")


if __name__ == "__main__":
    apply_labse_filter_optimized(
        input_file="DATA/fineTranlation/segmented_data_clean.jsonl",
        output_file="DATA/fineTranlation/segmented_data_clean_filtered.jsonl",
        batch_size=100000,
        model_batch_size=512,
        default_threshold=0.8
    )

    

# LSH

In [ ]:
import json
import re
from tqdm import tqdm
from datasketch import MinHash, MinHashLSH

# --- CẤU HÌNH ---
input_file = "DATA/fineTranlation/segmented_data_clean.jsonl"
output_file = "DATA/fineTranlation/segmented_data_deduplicated.jsonl"

LSH_THRESHOLD = 0.8  # Độ tương đồng (0.8 = 80%). Giảm xuống nếu muốn lọc mạnh hơn.
NUM_PERM = 128       # Số lượng hoán vị (càng cao càng chính xác nhưng chậm hơn)
N_GRAM = 3           # Kích thước shingle (3 từ liên tiếp)

def get_shingles(text, n=N_GRAM):
    """Tiền xử lý và chia văn bản thành các n-grams."""
    # Làm sạch: viết thường và bỏ ký tự đặc biệt
    text = re.sub(r'[^\w\s]', '', text.lower())
    words = text.split()
    if len(words) < n:
        return set(words)
    return set([" ".join(words[i:i+n]) for i in range(len(words) - n + 1)])

def create_minhash(text):
    """Tạo signature MinHash cho văn bản."""
    m = MinHash(num_perm=NUM_PERM)
    for shingle in get_shingles(text):
        m.update(shingle.encode('utf8'))
    return m

def run_deduplication():
    # Khởi tạo LSH
    lsh = MinHashLSH(threshold=LSH_THRESHOLD, num_perm=NUM_PERM)
    
    count_total = 0
    count_duplicates = 0
    
    # Đếm tổng số dòng để hiển thị thanh tiến trình (tùy chọn)
    print("Đang đếm số lượng dòng...")
    with open(input_file, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for _ in f)

    print(f"Bắt đầu xử lý {total_lines} dòng...")
    
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        for i, line in enumerate(tqdm(f_in, total=total_lines)):
            try:
                data = json.loads(line.strip())
                # Bạn có thể chọn lọc dựa trên 'source', 'target' hoặc cả hai:
                # Ví dụ lọc dựa trên câu nguồn:
                text_to_check = data['target'] 
                
                # 1. Tạo MinHash
                m = create_minhash(text_to_check)
                
                # 2. Truy vấn xem đã tồn tại câu tương tự chưa
                # Chúng ta dùng chỉ số dòng 'i' làm định danh duy nhất (key)
                result = lsh.query(m)
                
                if len(result) > 0:
                    # Tìm thấy trùng lặp gần giống (near-duplicate)
                    count_duplicates += 1
                    continue
                
                # 3. Nếu không trùng, lưu vào LSH và ghi ra file output
                lsh.insert(f"id_{i}", m)
                f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
                count_total += 1
                
            except Exception as e:
                print(f"Lỗi tại dòng {i}: {e}")
                continue

    print("\n--- KẾT QUẢ ---")
    print(f"Tổng số dòng đã đọc: {total_lines}")
    print(f"Số dòng bị loại bỏ (trùng lặp): {count_duplicates}")
    print(f"Số dòng giữ lại (duy nhất): {count_total}")
    print(f"Dữ liệu sạch đã được lưu tại: {output_file}")

if __name__ == "__main__":
    run_deduplication()

# Domain cluster

In [ ]:
import os

import json
import os
import torch
import numpy as np
import hdbscan
from transformers import AutoModel, AutoTokenizer
from pyvi import ViTokenizer
from tqdm import tqdm


# ==========================================
# CẤU HÌNH THAM SỐ
# ==========================================
INPUT_FILE = 'DATA/fineTranlation/clustered_results_opm1/cluster_noise_-1.jsonl'           # Đường dẫn file jsonl đầu vào
OUTPUT_DIR = 'DATA/fineTranlation/clustered_results'    # Thư mục chứa kết quả
N = 1000               # Kích thước cụm tối thiểu của HDBSCAN

# Tính năng mới
USE_UMAP = True                     # True: Sử dụng UMAP giảm chiều trước khi gom cụm, False: Chỉ dùng HDBSCAN
BATCH_SIZE = 256                     # Số lượng câu xử lý cùng lúc (Tăng/giảm tùy thuộc vào RAM/VRAM của máy)

if USE_UMAP:
    try:
        import umap
    except ImportError:
        raise ImportError("Vui lòng cài đặt umap bằng lệnh: pip install umap-learn")

def main():
    # 1. Đọc dữ liệu từ file jsonl
    data = []
    print("1. Đang đọc dữ liệu...")
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {INPUT_FILE}")
        return

    targets = [item['target'] for item in data]

    from sklearn.preprocessing import normalize

    print("3. Đang tải mô hình embedding...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"   => Thiết bị: {device}")

    MODEL_NAME = "intfloat/multilingual-e5-base"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    model.eval()

    print(f"4. Đang trích xuất embeddings (Batch size: {BATCH_SIZE})...")
    all_embeddings = []

    for i in tqdm(range(0, len(targets), BATCH_SIZE), desc="Extracting Embeddings"):
        batch_texts = targets[i:i+BATCH_SIZE]

        # E5 cần prefix
        batch_texts = [f"query: {text}" for text in batch_texts]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        last_hidden = outputs.last_hidden_state
        attention_mask = inputs["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()

        summed = torch.sum(last_hidden * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)

        batch_embeddings = (summed / counts).cpu().numpy()
        all_embeddings.extend(batch_embeddings)

    embeddings = np.array(all_embeddings)

    # normalize rất quan trọng
    embeddings = normalize(embeddings)

    print("\n[Debug] Đã thoát khỏi vòng lặp trích xuất.", flush=True)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    print("[Debug] Đang lưu ma trận embeddings ra file...", flush=True)
    np.save('DATA/fineTranlation/saved_embeddings_215k.npy', embeddings)
    print(f"[Debug] Đã tạo xong numpy array. Kích thước: {embeddings.shape}", flush=True)


    embeddings = np.load('DATA/fineTranlation/saved_embeddings_215k.npy')
    # 5. Giảm chiều dữ liệu bằng UMAP (Tùy chọn)
    if USE_UMAP:
        print("\n[Debug] Bắt đầu khởi tạo UMAP...", flush=True)
        reducer = umap.UMAP(
            n_neighbors=0.005*N,
            n_components=5,
            min_dist=0.0,
            metric='cosine',
            verbose=True,
            n_epochs=500
        )
        print("[Debug] Đang đưa dữ liệu vào UMAP (Quá trình này có thể tốn vài phút JIT Compile)...", flush=True)
        embeddings = reducer.fit_transform(embeddings)
        print("[Debug] UMAP hoàn tất!", flush=True)
    else:
        print("5. Bỏ qua bước UMAP.", flush=True)

    # 6. Phân cụm bằng HDBSCAN
    print("6. Đang phân cụm bằng HDBSCAN...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=0.01*N,
        min_samples=0.005*N,
        metric='euclidean',
        #cluster_selection_method='eom',
        #cluster_selection_epsilon=0.1,
        #prediction_data=True
    )
    cluster_labels = clusterer.fit_predict(embeddings)

    # 7. Phân nhóm và xuất file
    print("7. Đang lưu kết quả ra thư mục...")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    clustered_data = {}
    for item, label in zip(data, cluster_labels):
        if label not in clustered_data:
            clustered_data[label] = []
        clustered_data[label].append(item)

    for label, items in tqdm(clustered_data.items(), desc="Writing Files"):
        filename = "cluster_noise.jsonl" if label == -1 else f"cluster_{label}.jsonl"
        filepath = os.path.join(OUTPUT_DIR, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            for item in items:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print("-" * 50)
    print(f"✅ Hoàn thành! Đã lưu kết quả tại thư mục: '{OUTPUT_DIR}'")
    print(f"📊 Thống kê: Tìm thấy {len(clustered_data.keys())} cụm (tính cả cụm nhiễu -1).")

if __name__ == "__main__":
    main()